In [30]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

In [7]:
DATA_DIR = Path('../data/')
MAN_DIR = DATA_DIR / 'manifests'
RAW_DIR = DATA_DIR / 'raw'
XLSX_PATH = RAW_DIR / 'pat_info.xlsx'
PAR_PATH = MAN_DIR / 'image_manifest.parquet'

In [20]:
df = pd.read_parquet(PAR_PATH)
df.head(10)

,patient_id,cohort,severity_folder,pose_index,modality,filepath,hb_grade,Side,Gender,Age
0,CompleteFlaccid1,Flaccid,CompleteFlaccid,1.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0
1,CompleteFlaccid1,Flaccid,CompleteFlaccid,2.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0
2,CompleteFlaccid1,Flaccid,CompleteFlaccid,3.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0
3,CompleteFlaccid1,Flaccid,CompleteFlaccid,4.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0
4,CompleteFlaccid1,Flaccid,CompleteFlaccid,5.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0
5,CompleteFlaccid1,Flaccid,CompleteFlaccid,6.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0
6,CompleteFlaccid1,Flaccid,CompleteFlaccid,7.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0
7,CompleteFlaccid1,Flaccid,CompleteFlaccid,8.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0
8,CompleteFlaccid1,Flaccid,CompleteFlaccid,NaN,video,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0
9,CompleteFlaccid2,Flaccid,CompleteFlaccid,1.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Right,Female,16.0


In [51]:
patient_df = df[
    ["patient_id", "hb_grade", "cohort", "severity_folder"]
].drop_duplicates()
len(patient_df)

60

In [29]:
print(patient_df.shape)
print(patient_df["patient_id"].nunique())
print(patient_df["hb_grade"].value_counts().sort_index())

(60, 4)
60
hb_grade
1    10
2    10
3    10
4    10
5    10
6    10
Name: count, dtype: int64


In [32]:
train_patients, temp_patients = train_test_split(
    patient_df,
    test_size=0.30,
    stratify=patient_df["hb_grade"],
    random_state=42
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    stratify=temp_patients["hb_grade"],
    random_state=42
)

pandas.DataFrame

In [34]:
train_patients = train_patients.copy()
val_patients = val_patients.copy()
test_patients = test_patients.copy()

train_patients["split"] = "train"
val_patients["split"] = "val"
test_patients["split"] = "test"

patient_splits = pd.concat(
    [train_patients, val_patients, test_patients],
    ignore_index=True
)

In [35]:
print(patient_splits["split"].value_counts())
print(pd.crosstab(patient_splits["hb_grade"], patient_splits["split"]))

split
train    42
val       9
test      9
Name: count, dtype: int64
split     test  train  val
hb_grade                  
1            1      7    2
2            1      7    2
3            2      7    1
4            2      7    1
5            2      7    1
6            1      7    2


In [36]:
dup_counts = patient_splits["patient_id"].value_counts()
print(dup_counts[dup_counts > 1])

Series([], Name: count, dtype: int64)


In [38]:
df_with_splits = df.merge(
    patient_splits[["patient_id", "split"]],
    on="patient_id",
    how="left"
)

In [41]:
print(df_with_splits["split"].isna().sum())

0


In [44]:
train_ids = set(df_with_splits.loc[df_with_splits["split"] == "train", "patient_id"])
val_ids = set(df_with_splits.loc[df_with_splits["split"] == "val", "patient_id"])
test_ids = set(df_with_splits.loc[df_with_splits["split"] == "test", "patient_id"])

print(train_ids & val_ids)
print(train_ids & test_ids)
print(val_ids & test_ids)

set()
set()
set()


In [48]:
df_test = pd.read_parquet(MAN_DIR / 'image_manifest_with_splits.parquet')
df_test.head(20)

,patient_id,cohort,severity_folder,pose_index,modality,filepath,hb_grade,Side,Gender,Age,split
0,CompleteFlaccid1,Flaccid,CompleteFlaccid,1.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0,val
1,CompleteFlaccid1,Flaccid,CompleteFlaccid,2.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0,val
2,CompleteFlaccid1,Flaccid,CompleteFlaccid,3.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0,val
3,CompleteFlaccid1,Flaccid,CompleteFlaccid,4.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0,val
4,CompleteFlaccid1,Flaccid,CompleteFlaccid,5.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0,val
5,CompleteFlaccid1,Flaccid,CompleteFlaccid,6.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0,val
6,CompleteFlaccid1,Flaccid,CompleteFlaccid,7.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0,val
7,CompleteFlaccid1,Flaccid,CompleteFlaccid,8.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0,val
8,CompleteFlaccid1,Flaccid,CompleteFlaccid,NaN,video,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Left,Male,54.0,val
9,CompleteFlaccid2,Flaccid,CompleteFlaccid,1.0,image,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,Right,Female,16.0,train
